Konteks / Skenario

Kalian direkrut sebagai data analyst paruh waktu oleh "Kopi Nusantara", jaringan kedai kopi dengan beberapa cabang. Pemilik ingin memahami performa penjualan tiap cabang, menu terlaris, dan kontribusi barista kepala di masing-masing cabang selama periode tertentu — persis seperti studi kasus yang baru dikerjakan bersama, namun dengan data dan struktur yang sedikit berbeda.

In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON).
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV).
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV).
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

# Upload ke HDFS.
!hdfs dfs -mkdir -p /user/caitlyn/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/caitlyn/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/caitlyn/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/caitlyn/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [2]:
# SparkSession.
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, avg, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Tugas7_Naufal").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 05:55:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/08 05:55:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


A. EXTRACT (bobot 10%)

Baca ketiga sumber data dari HDFS (bukan disk lokal) menjadi tiga Spark DataFrame.

In [3]:
# Extract dari HDFS.
df_trx = spark.read.csv("hdfs://localhost:9000/user/caitlyn/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json("hdfs://localhost:9000/user/caitlyn/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/caitlyn/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

B. TRANSFORM — Join & Kolom Turunan (bobot 20%)

Gabungkan ketiga tabel, tambahkan kolom total_penjualan (qty x harga).

In [4]:
# Transformasi Join dan penambahan kolom.
df_gabung = df_trx.join(df_cabang, on="cabang_id", how="inner") \
                  .join(df_menu, on="menu_id", how="inner")

df_gabung = df_gabung.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabung.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


C. TRANSFORM — Window Function (bobot 20%)

Tentukan top-2 menu terlaris (berdasarkan total_penjualan) di setiap cabang, gunakan row_number().

In [5]:
# Agregasi penjualan per menu di tiap cabang terlebih dulu.
ringkasan_menu_cabang = df_gabung.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# Ambil top 2 menggunakan window function partisi cabang.
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

top2_menu = ringkasan_menu_cabang.withColumn("rn", row_number().over(window_cabang)) \
    .filter(col("rn") <= 2) \
    .drop("rn")

top2_menu.orderBy("nama_cabang", col("total_penjualan").desc()).show()

+--------------------+---------+---------------+
|         nama_cabang|nama_menu|total_penjualan|
+--------------------+---------+---------------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|
|Kopi Nusantara Ma...|  Menu-12|        8085000|
|Kopi Nusantara Se...|  Menu-13|        8750000|
|Kopi Nusantara Se...|   Menu-9|        8715000|
|Kopi Nusantara Yo...|  Menu-12|        9275000|
|Kopi Nusantara Yo...|  Menu-15|        8160000|
+--------------------+---------+---------------+



D. TRANSFORM — Spark SQL (bobot 20%)

Menggunakan Spark SQL murni, tampilkan total penjualan per kepala_barista, diurutkan dari tertinggi.

In [6]:
df_gabung.createOrReplaceTempView("kopi_nusantara")

spark.sql("""
    SELECT kepala_barista, SUM(total_penjualan) AS total_kontribusi
    FROM kopi_nusantara
    GROUP BY kepala_barista
    ORDER BY total_kontribusi DESC
""").show()

+--------------+----------------+
|kepala_barista|total_kontribusi|
+--------------+----------------+
|         Nadia|       177680000|
|          Reza|        84645000|
+--------------+----------------+



E. LOAD (bobot 15%)

Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, path /user/[username]/tugas7/processed/laporan, dipartisi berdasarkan kategori_menu. Verifikasi dengan membaca kembali dan menghitung jumlah baris.

In [7]:
path_load = "hdfs://localhost:9000/user/caitlyn/tugas7/processed/laporan"

# Menyimpan dalam bentuk Parquet berpartisi.
df_gabung.write.mode("overwrite").partitionBy("kategori_menu").parquet(path_load)

# Verifikasi pembacaan ulang.
df_verifikasi = spark.read.parquet(path_load)
print("Data berhasil disimpan dengan total baris:", df_verifikasi.count())

Data berhasil disimpan dengan total baris: 4000


F. Ringkasan Eksekutif & Rekomendasi (bobot 15%)

Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi), lalu tulis pada markdown cell (minimal 120 kata): cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.

In [8]:
# Membuat tabel ringkasan.
df_ringkasan = df_gabung.groupBy("nama_cabang", "kota").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

df_ringkasan.show()

+--------------------+----------+---------------+----------------+-------------------------+
|         nama_cabang|      kota|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|       84645000|            1305|        64862.06896551724|
+--------------------+----------+---------------+----------------+-------------------------+



Ringkasan Eksekutif & Rekomendasi
Berdasarkan hasil olahan tabel ringkasan eksekutif, cabang dengan kinerja terbaik secara keseluruhan diduduki oleh Kopi Nusantara Semarang. Cabang tersebut mencetak nilai metrik tertinggi pada total penjualan 98.542.000 dan jumlah transaksi trebanyak - yaitu 1.393 transaksi. Sebaliknya, cabang yang berada di peringkat paling bawah dan memerlukan perhatian khusus dari manajemen adalah Kopi Nusantara Yogyakarta dengan total penjualan terendah sebesar 84.645.000, serta rata-rata nilai transaksi yang paling kecil dibandingkan cabang lainnya.
Untuk mengoptimalkan cabang Yogyakarta yang berkinerja terlemah, pihak manajemen perlu melakukan intervensi strategi promosi yang spesifik pada area tersebut. Mengacu pada hasil analitik window function di Bagian C yang melacak performa top-2 menu terlaris pada masing-masing cabang, strategi terbaik yang dapat diambil adalah menggencarkan promosi secara masif pada menu-menu andalan tersebut. Memberikan paket bundling atau diskon khusus untuk menu terlaris di cabang terlemah akan bertindak sebagai loss leader (daya tarik utama yang terbukti diminati pasar lokal). Hal ini merupakan metode yang efektif untuk memancing lonjakan traffic pelanggan baru, yang pada akhirnya akan mendongkrak metrik jumlah_transaksi dan rata_rata_nilai_transaksi di cabang tersebut agar dapat menyusul performa dari cabang-cabang lainnya.